### 1. Importing Libraries
Imports essential libraries for tensor/matrix manipulations (`numpy`), data handling (`pandas`), performance metrics (`sklearn`), and dataset downloading (`tensorflow.keras.datasets`).

In [69]:
import numpy as np

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.datasets import mnist

### 2. Loading & Normalizing the MNIST Dataset
- Automatically downloads the MNIST dataset containing 60,000 training images and 10,000 test images.
- Flattens each $28 \times 28$ pixel image into a 1D vector of shape `(784,)` and normalizes pixel values to a range of `[0, 1]` by dividing by `255.0`.

In [70]:
# This downloads MNIST automatically from a stable mirror
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Flatten and normalize the pixels for your scratch network
X_train = X_train.reshape(X_train.shape[0], -1) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0

print("Train shape:", X_train.shape)  # (60000, 784)
print("Labels shape:", y_train.shape)   # (60000,)

Train shape: (60000, 784)
Labels shape: (60000,)


### 3. Data Inspection & Dimensions
Quickly checks the dataset shapes and sizes to ensure the feature matrix and labels match expectations ($m = 60000$ training examples).


In [71]:
X_train.shape


(60000, 784)

In [72]:
y_train.size

60000

### 4. One-Hot Encoding Target Labels
Converts the 1D array of integer labels (`y_train`) into a 2D one-hot encoded matrix of shape `(60000, 10)` using advanced NumPy indexing.

In [73]:
def convert_to_one_hot(y, num_classes=10):
    m = y.size
    one_hot = np.zeros((m, num_classes))  # Shape: (60000, 10)
    one_hot[np.arange(m), y] = 1
    return one_hot

# Example usage:
# y_train is your raw integer labels array (shape: (60000,))
Y_train_encoded = convert_to_one_hot(y_train)

print(Y_train_encoded.shape)


(60000, 10)


### 5. Weight Initialization (He Initialization)
- Configures a 3-layer neural network with layer dimensions: `[784, 25, 15, 10]`.
- Uses **He Initialization** ($\text{randn} \times \sqrt{2.0 / n\_row}$) to scale weights and prevent vanishing or exploding gradients during ReLU activation.
- Initializes biases to zeros.

In [74]:
def n_layer(layers):
    length = len(layers)
    parameters = {}
    for l in range(1,length):
        n_row = layers[l-1]

        n_col = layers[l]
        parameters["W" + str(l)] = np.random.rand(n_row,n_col)* np.sqrt(2.0 / n_row)  #normalizing weights
        parameters["b" + str(l)] = np.zeros((1,n_col))


    return parameters
parms = n_layer([X_train.shape[1],25,15,10])
print(parms["W1"].shape)
print(parms["W2"].shape)
print(parms["W3"].shape)


(784, 25)
(25, 15)
(15, 10)


### 6. Cost Function — Categorical Cross-Entropy
- Computes cross-entropy loss between true one-hot labels and predicted probabilities.
- Implements numerical stability clipping (`eps = 1e-15`) to prevent `log(0)` calculation errors.

In [75]:
def cross_entropy(Y_true, A3):
    m = Y_true.shape[0]


    eps = 1e-15
    A3_clipped = np.clip(A3, eps, 1.0 - eps)


    loss = -np.sum(Y_true * np.log(A3_clipped)) / m

    return loss


### 7. Softmax Activation Function
Computes output probabilities across class columns using the max trick (`axis=1`, `keepdims=True`) for numerical overflow protection.

In [76]:
def softmax(Z):
    shift_Z = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(shift_Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

### 8. ReLU Activation Function
Applies the Rectified Linear Unit ($\max(0, z)$) element-wise across the hidden layer pre-activations.

In [77]:
 def Relu(z):
    return np.maximum(0,z)


### 9. Forward Propagation
Sequentially computes linear transformations ($Z = XW + b$) and non-linear activations through all layers (`Layer 1 -> Layer 2 -> Layer 3`) to yield final output probabilities (`A3`).

In [78]:
def forward_prop(parameters,X):
     W1 = parameters["W1"]
     b1 = parameters["b1"]

     W2 = parameters["W2"]
     b2 = parameters["b2"]

     W3 = parameters["W3"]
     b3 = parameters["b3"]

     z1 = X @ W1 + b1
     A1 = Relu(z1)

     z2 = A1 @ W2 + b2
     A2 = Relu(z2)

     z3 = A2 @ W3 + b3
     A3 = softmax(z3)

     return z1,z2,z3,A1,A2,A3



In [79]:
z1,z2,z3,A1,A2,A3 = forward_prop(parms, X_train)
A3


array([[6.50336099e-04, 3.59476378e-05, 4.64724323e-04, ...,
        7.74626991e-06, 7.13825881e-03, 1.27656549e-03],
       [2.19104744e-04, 8.16341728e-06, 1.53086831e-04, ...,
        1.45797972e-06, 3.58480104e-03, 5.09492387e-04],
       [4.59718870e-03, 6.03743097e-04, 3.83184927e-03, ...,
        2.17358374e-04, 2.57829005e-02, 7.96347315e-03],
       ...,
       [2.89720658e-03, 2.91173811e-04, 2.25636631e-03, ...,
        9.17113261e-05, 1.84722755e-02, 4.74060426e-03],
       [4.09421720e-03, 4.74459442e-04, 3.18897744e-03, ...,
        1.66263876e-04, 2.41707010e-02, 6.97388035e-03],
       [3.51534571e-03, 3.87209805e-04, 2.73208595e-03, ...,
        1.21956882e-04, 2.13353821e-02, 5.92232691e-03]],
      shape=(60000, 10))

### 10. Backward Propagation
- Computes output layer errors (`dz3 = A3 - Y`) and propagates gradients backward through the network (`dz2 -> dz1`).
- Calculates weight and bias gradients (`dW`, `db`) for all layers and updates parameters using Gradient Descent.

In [80]:
def backwardpropogation(params, X, Y, z1, z2, A1, A2, A3, learning_rate=0.01):
    m = Y.shape[0]

    # Output Layer Gradients (Layer 3)
    dz3 = A3 - Y
    dW3 = (A2.T @ dz3) / m
    db3 = np.sum(dz3, axis=0, keepdims=True) / m

    # Hidden Layer 2 Gradients
    dA2 = dz3 @ params["W3"].T
    dz2 = dA2 * (z2 > 0)
    dW2 = (A1.T @ dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    # Hidden Layer 1 Gradients
    dA1 = dz2 @ params["W2"].T
    dz1 = dA1 * (z1 > 0)
    dW1 = (X.T @ dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    # Update Parameters using Gradient Descent
    params["W1"] -= learning_rate * dW1
    params["b1"] -= learning_rate * db1
    params["W2"] -= learning_rate * dW2
    params["b2"] -= learning_rate * db2
    params["W3"] -= learning_rate * dW3
    params["b3"] -= learning_rate * db3

    return params

In [81]:
z1,z2,z3,A1,A2,A3 = forward_prop(params, X_train)
print(A3.shape)
print(Y_train_encoded.shape)

(60000, 10)
(60000, 10)


### 11. Model Training & Evaluation
- Executes **Mini-Batch Gradient Descent** across multiple epochs, utilizing data shuffling, batch slicing, and **learning rate decay** to stably minimize cross-entropy loss and achieve ultra-high precision.
- Evaluates model performance on unseen test data (`X_test`), extracting predictions via `np.argmax` across output class columns to calculate final test set classification accuracy.

In [82]:
def training_minibatch_decay(X, Y, initial_lr=0.1, epochs= 50, batch_size=64):
    params = n_layer([X.shape[1], 25, 15, 10])
    m = X.shape[0]

    for epoch in range(epochs):
        # Shuffling the  dataset
        permutation = np.random.permutation(m)
        X_shuffled = X[permutation]
        Y_shuffled = Y[permutation]

        # Decay learning rate over time gets smaller each epoch
        learning_rate = initial_lr / (1.0 + 0.05 * epoch)

        #Mini-batch loop
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i : i + batch_size]
            Y_batch = Y_shuffled[i : i + batch_size]

            z1, z2, z3, A1, A2, A3 = forward_prop(params, X_batch)
            params = backwardpropogation(params, X_batch, Y_batch, z1, z2, A1, A2, A3, learning_rate)


        _, _, _, _, _, A3_full = forward_prop(params, X)
        loss = cross_entropy(Y, A3_full)
        print(f"Epoch {epoch+1}/{epochs} | LR: {learning_rate:.4f} | Loss: {loss:.5f}")

    return params

params = training_minibatch_decay(X_train, Y_train_encoded)

Epoch 1/50 | LR: 0.1000 | Loss: 0.38336
Epoch 2/50 | LR: 0.0952 | Loss: 0.25351
Epoch 3/50 | LR: 0.0909 | Loss: 0.19972
Epoch 4/50 | LR: 0.0870 | Loss: 0.16294
Epoch 5/50 | LR: 0.0833 | Loss: 0.15574
Epoch 6/50 | LR: 0.0800 | Loss: 0.14382
Epoch 7/50 | LR: 0.0769 | Loss: 0.13133
Epoch 8/50 | LR: 0.0741 | Loss: 0.12601
Epoch 9/50 | LR: 0.0714 | Loss: 0.11725
Epoch 10/50 | LR: 0.0690 | Loss: 0.11042
Epoch 11/50 | LR: 0.0667 | Loss: 0.10301
Epoch 12/50 | LR: 0.0645 | Loss: 0.10382
Epoch 13/50 | LR: 0.0625 | Loss: 0.10772
Epoch 14/50 | LR: 0.0606 | Loss: 0.09039
Epoch 15/50 | LR: 0.0588 | Loss: 0.08864
Epoch 16/50 | LR: 0.0571 | Loss: 0.09303
Epoch 17/50 | LR: 0.0556 | Loss: 0.09234
Epoch 18/50 | LR: 0.0541 | Loss: 0.08001
Epoch 19/50 | LR: 0.0526 | Loss: 0.08382
Epoch 20/50 | LR: 0.0513 | Loss: 0.08455
Epoch 21/50 | LR: 0.0500 | Loss: 0.07499
Epoch 22/50 | LR: 0.0488 | Loss: 0.07907
Epoch 23/50 | LR: 0.0476 | Loss: 0.07522
Epoch 24/50 | LR: 0.0465 | Loss: 0.07207
Epoch 25/50 | LR: 0.0455 

In [83]:
# Run forward propagation on the test data
z1, z2, z3, A1, A2, A3 = forward_prop(params, X_test)

# Find the index of the highest probability for each prediction (0 to 9)
predictions = np.argmax(A3, axis=1)

# Compare predictions against true test labels
accuracy = np.mean(predictions == y_test) * 100

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.20%


### 12. Comparison: Custom NumPy vs. TensorFlow Keras Model
- **Custom NumPy Model**: Achieved a test accuracy of **96.20%** using your scratch-built forward pass, backpropagation, and mini-batch gradient descent.
- **TensorFlow Keras Model**: Achieved a training accuracy of **96.27%** (loss: `0.1248`) after 5 epochs using the built-in Sequential API and Adam optimizer.
- **Conclusion**: Both implementations achieve nearly identical performance (~96.2%), proving that my custom neural network built completely from scratch in NumPy matches the accuracy of industry-standard frameworks!

In [84]:
model = Sequential([
    Dense(25, activation = 'relu'),
    Dense(15, activation = 'relu'),
    Dense(10, activation = 'softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8862 - loss: 0.3902
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9425 - loss: 0.1952
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9521 - loss: 0.1610
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9601 - loss: 0.1386
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9627 - loss: 0.1248
